# CCCD (ID Card) Image Processing Pipeline

Pipeline (demo 1 ảnh đầu tiên):
- Input image
- Image Quality Check
- Document Detection / Crop CCCD (perspective warp)
- Image Preprocessing
- Field Localization (vẽ ROI theo template)

Ghi chú: Notebook này tập trung vào **xử lý ảnh**. Bước OCR sẽ được nối sau (PaddleOCR/Tesseract) tùy môi trường.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "scripts").exists():
            return p
    return start


DOC_TYPE = "id_card"
RUN_DATE = "2026-05-19"

# Usually Jupyter starts in workspace root, but this keeps it robust.
PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "unstructured"
    / "documents"
    / f"doc_type={DOC_TYPE}"
    / f"run_date={RUN_DATE}"
)

assert DATA_DIR.exists(), f"Không thấy input dir: {DATA_DIR}"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)


In [ ]:
# --- Load first image (recursively) ---
img_paths: list[Path] = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.tif", "*.tiff"):
    img_paths.extend(DATA_DIR.rglob(ext))
img_paths = sorted(img_paths)

assert img_paths, f"Không tìm thấy ảnh trong: {DATA_DIR}"
img_path = img_paths[0]
print("Using image:", img_path)

bgr = cv2.imread(str(img_path))
assert bgr is not None, f"cv2.imread failed: {img_path}"
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))
plt.imshow(rgb)
plt.axis("off")
plt.title("Input image")
plt.show()


In [ ]:
# --- Deskew Input Image (handle small/medium tilt before document detection) ---

def rotate_bound_bgr(bgr: np.ndarray, angle_degrees: float, border_value: tuple[int, int, int] = (255, 255, 255)) -> np.ndarray:
    h, w = bgr.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)

    cos = abs(M[0, 0])
    sin = abs(M[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    M[0, 2] += (new_w / 2.0) - center[0]
    M[1, 2] += (new_h / 2.0) - center[1]

    return cv2.warpAffine(
        bgr,
        M,
        (new_w, new_h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )


def _estimate_angle_from_edges(edges: np.ndarray, min_len: int) -> float | None:
    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=50,
        minLineLength=min_len,
        maxLineGap=20,
    )
    if lines is None:
        return None

    angles: list[float] = []
    weights: list[float] = []
    for line in lines[:, 0]:
        x1, y1, x2, y2 = [int(v) for v in line]
        dx = x2 - x1
        dy = y2 - y1
        length = float(np.hypot(dx, dy))
        if length < min_len:
            continue

        angle = float(np.degrees(np.arctan2(dy, dx)))
        angle = ((angle + 90.0) % 180.0) - 90.0
        if angle > 45.0:
            angle -= 90.0
        elif angle < -45.0:
            angle += 90.0

        if abs(angle) <= 30.0:
            angles.append(angle)
            weights.append(length)

    if not angles:
        return None

    order = np.argsort(angles)
    sorted_angles = np.asarray(angles, dtype=np.float32)[order]
    sorted_weights = np.asarray(weights, dtype=np.float32)[order]
    cumsum = np.cumsum(sorted_weights)
    midpoint = float(cumsum[-1] * 0.5)
    return float(sorted_angles[int(np.searchsorted(cumsum, midpoint))])


def estimate_skew_angle_degrees(bgr: np.ndarray) -> float | None:
    h, w = bgr.shape[:2]
    min_len = max(80, int(min(h, w) * 0.25))

    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    blue_mask = cv2.inRange(hsv, np.array([85, 45, 20]), np.array([135, 255, 190]))
    blue_mask = cv2.morphologyEx(blue_mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8), iterations=2)
    blue_edges = cv2.Canny(blue_mask, 50, 150)
    blue_angle = _estimate_angle_from_edges(blue_edges, min_len=max(40, int(min(h, w) * 0.18)))
    if blue_angle is not None:
        return blue_angle

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(gray, 50, 150)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)
    return _estimate_angle_from_edges(edges, min_len=min_len)


def deskew_input_image(bgr: np.ndarray, min_abs_angle: float = 0.75) -> tuple[np.ndarray, float, dict[str, float | None]]:
    estimated = estimate_skew_angle_degrees(bgr)
    if estimated is None or abs(estimated) < min_abs_angle:
        return bgr, 0.0, {"estimated_skew": estimated, "residual_skew": estimated}

    candidates: list[tuple[float, np.ndarray, float | None]] = [(0.0, bgr, estimated)]
    for rotation in (-estimated, estimated):
        rotated = rotate_bound_bgr(bgr, rotation)
        residual = estimate_skew_angle_degrees(rotated)
        candidates.append((rotation, rotated, residual))

    best_rotation, best_image, best_residual = min(
        candidates,
        key=lambda item: abs(item[2]) if item[2] is not None else 999.0,
    )
    return best_image, float(best_rotation), {
        "estimated_skew": estimated,
        "residual_skew": best_residual,
    }


bgr, deskew_degrees, deskew_scores = deskew_input_image(bgr)
print("Deskew applied:", deskew_degrees)
print("Deskew scores:", deskew_scores)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Deskewed input (rotation={deskew_degrees:.2f}°)")
plt.show()


In [ ]:
# --- Image Quality Check ---

def image_quality_metrics(bgr: np.ndarray) -> dict:
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    # Blur score: variance of Laplacian
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    blur_var = float(lap.var())

    # Brightness/contrast
    brightness = float(gray.mean())
    contrast = float(gray.std())

    # Simple saturation check (fraction near white/black)
    pct_black = float((gray < 10).mean())
    pct_white = float((gray > 245).mean())

    return {
        "width": w,
        "height": h,
        "blur_var_laplacian": blur_var,
        "brightness_mean": brightness,
        "contrast_std": contrast,
        "pct_black": pct_black,
        "pct_white": pct_white,
    }

metrics = image_quality_metrics(bgr)
metrics


In [ ]:
# --- Document Detection / Crop CCCD (find quad + perspective warp) ---

def _order_points_clockwise(pts: np.ndarray) -> np.ndarray:
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # tl
    rect[2] = pts[np.argmax(s)]  # br
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # tr
    rect[3] = pts[np.argmax(diff)]  # bl
    return rect


def _quad_aspect_ratio(pts: np.ndarray) -> float:
    rect = _order_points_clockwise(pts)
    tl, tr, br, bl = rect
    width = max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl))
    height = max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl))
    return float(max(width, height) / max(1.0, min(width, height)))


def _is_plausible_card_quad(pts: np.ndarray, area_img: float, min_area_ratio: float = 0.25) -> bool:
    area_ratio = cv2.contourArea(pts.astype(np.float32)) / max(1.0, area_img)
    aspect = _quad_aspect_ratio(pts)
    return area_ratio >= min_area_ratio and 1.25 <= aspect <= 1.95


def _find_quad_from_edges(edges: np.ndarray, area_img: float, offset_xy: tuple[int, int] = (0, 0)) -> tuple[np.ndarray | None, dict]:
    cnts, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
    debug: dict = {
        "candidate_areas": [float(cv2.contourArea(c) / max(1.0, area_img)) for c in cnts[:8]]
    }
    ox, oy = offset_xy

    for c in cnts[:40]:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2).astype(np.float32)
            pts[:, 0] -= ox
            pts[:, 1] -= oy
            if _is_plausible_card_quad(pts, area_img):
                debug["quad_method"] = "approx"
                debug["quad_area_ratio"] = float(cv2.contourArea(pts) / max(1.0, area_img))
                return _order_points_clockwise(pts), debug

    for c in cnts[:40]:
        if cv2.contourArea(c) <= 0:
            continue

        rect = cv2.minAreaRect(c)
        box = cv2.boxPoints(rect).astype(np.float32)
        box[:, 0] -= ox
        box[:, 1] -= oy
        if _is_plausible_card_quad(box, area_img):
            debug["quad_method"] = "min_area_rect"
            debug["quad_area_ratio"] = float(cv2.contourArea(box) / max(1.0, area_img))
            return _order_points_clockwise(box), debug

    return None, debug


def find_document_quad(bgr: np.ndarray) -> tuple[np.ndarray | None, dict]:
    h, w = bgr.shape[:2]
    pad = max(20, int(min(h, w) * 0.04))
    padded = cv2.copyMakeBorder(
        bgr,
        pad,
        pad,
        pad,
        pad,
        cv2.BORDER_CONSTANT,
        value=(255, 255, 255),
    )
    gray = cv2.cvtColor(padded, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(gray, 35, 120)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8), iterations=2)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)

    area_img = float(h * w)
    quad, debug = _find_quad_from_edges(edges, area_img=area_img, offset_xy=(pad, pad))
    debug["edges"] = edges[pad:pad + h, pad:pad + w]
    if quad is not None:
        debug["quad_method"] = f"padded_{debug.get('quad_method')}"
        return quad, debug

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(gray, 50, 150)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)
    debug["edges"] = edges

    quad, debug = _find_quad_from_edges(edges, area_img=area_img)
    debug["edges"] = edges
    if quad is not None:
        return quad, debug

    debug["quad_method"] = "none_large_enough"
    return None, debug


def warp_document(bgr: np.ndarray, quad: np.ndarray, out_w: int = 1000) -> np.ndarray:
    (tl, tr, br, bl) = quad

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxW = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxH = int(max(heightA, heightB))

    scale = out_w / max(1, maxW)
    out_h = int(maxH * scale)

    dst = np.array(
        [[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]],
        dtype=np.float32,
    )
    M = cv2.getPerspectiveTransform(quad, dst)
    warped = cv2.warpPerspective(bgr, M, (out_w, out_h))
    return warped


quad, dbg = find_document_quad(bgr)
print("Found quad:", quad is not None)
print("Quad method:", dbg.get("quad_method"))
print("Quad area ratio:", dbg.get("quad_area_ratio"))
print("Top contour area ratios:", dbg.get("candidate_areas"))

vis = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
if quad is not None:
    q = quad.astype(int)
    cv2.polylines(vis, [q], isClosed=True, color=(255, 0, 0), thickness=4)

plt.figure(figsize=(12, 6))
plt.imshow(vis)
plt.axis("off")
plt.title("Detected document contour (if any)")
plt.show()

plt.figure(figsize=(12, 4))
plt.imshow(dbg["edges"], cmap="gray")
plt.axis("off")
plt.title("Edges (Canny)")
plt.show()

if quad is None:
    warped = bgr
    print("Fallback: use original image (no quad found)")
else:
    warped = warp_document(bgr, quad, out_w=1000)

warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 6))
plt.imshow(warped_rgb)
plt.axis("off")
plt.title("Warped/Cropped document (normalized width=1000)")
plt.show()


In [ ]:
# --- Orientation Normalization (rotate card to upright) ---
# The DEMO front template has a dark blue title band.
# Detect whether that band is on top/bottom/left/right, rotate upright, then restore width=1000.

def normalize_width(bgr_doc: np.ndarray, out_w: int = 1000) -> np.ndarray:
    h, w = bgr_doc.shape[:2]
    if w == out_w:
        return bgr_doc

    out_h = max(1, int(h * (out_w / max(1, w))))
    return cv2.resize(bgr_doc, (out_w, out_h), interpolation=cv2.INTER_AREA)


def is_plausible_front_card_image(bgr_doc: np.ndarray) -> bool:
    h, w = bgr_doc.shape[:2]
    aspect = h / max(1, w)
    return 0.50 <= aspect <= 0.85


def normalize_front_orientation(bgr_doc: np.ndarray, out_w: int = 1000) -> tuple[np.ndarray, int, dict[str, float]]:
    hsv = cv2.cvtColor(bgr_doc, cv2.COLOR_BGR2HSV)
    blue_mask = cv2.inRange(hsv, np.array([85, 45, 20]), np.array([135, 255, 180]))

    h, w = blue_mask.shape
    band_h = max(1, int(h * 0.22))
    band_w = max(1, int(w * 0.22))

    side_scores = {
        "top_blue": float((blue_mask[:band_h] > 0).mean()),
        "bottom_blue": float((blue_mask[-band_h:] > 0).mean()),
        "left_blue": float((blue_mask[:, :band_w] > 0).mean()),
        "right_blue": float((blue_mask[:, -band_w:] > 0).mean()),
    }
    header_side = max(side_scores, key=side_scores.get)

    rotated = bgr_doc
    rotation_degrees = 0
    if side_scores[header_side] > 0.03:
        if header_side == "bottom_blue":
            rotated = cv2.rotate(bgr_doc, cv2.ROTATE_180)
            rotation_degrees = 180
        elif header_side == "left_blue":
            rotated = cv2.rotate(bgr_doc, cv2.ROTATE_90_CLOCKWISE)
            rotation_degrees = 90
        elif header_side == "right_blue":
            rotated = cv2.rotate(bgr_doc, cv2.ROTATE_90_COUNTERCLOCKWISE)
            rotation_degrees = 270

    return normalize_width(rotated, out_w=out_w), rotation_degrees, side_scores


def normalize_card_geometry_for_rois(bgr_doc: np.ndarray, out_w: int = 1000) -> tuple[np.ndarray, dict[str, float | None]]:
    normalized = normalize_width(bgr_doc, out_w=out_w)
    deskewed, deskew_degrees, deskew_scores = deskew_input_image(normalized, min_abs_angle=0.20)
    quad, quad_debug = find_document_quad(deskewed)
    if quad is not None:
        deskewed = warp_document(deskewed, quad, out_w=out_w)
    else:
        deskewed = normalize_width(deskewed, out_w=out_w)

    return deskewed, {
        "roi_deskew_degrees": deskew_degrees,
        "roi_estimated_skew": deskew_scores.get("estimated_skew"),
        "roi_residual_skew": deskew_scores.get("residual_skew"),
        "roi_quad_area_ratio": quad_debug.get("quad_area_ratio"),
    }


warped, rotation_degrees, orientation_scores = normalize_front_orientation(warped, out_w=1000)
if not is_plausible_front_card_image(warped):
    print("Warp result does not look like a full front card; fallback to deskewed input.")
    warped, rotation_degrees, orientation_scores = normalize_front_orientation(bgr, out_w=1000)

warped, roi_geometry_scores = normalize_card_geometry_for_rois(warped, out_w=1000)

print("Rotation applied:", rotation_degrees)
print("Orientation scores:", orientation_scores)
print("ROI geometry scores:", roi_geometry_scores)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Orientation-normalized document (rotation={rotation_degrees}°)")
plt.show()


In [ ]:
# --- Image Preprocessing ---

def preprocess_for_ocr(bgr_doc: np.ndarray) -> dict[str, np.ndarray]:
    out: dict[str, np.ndarray] = {}
    gray = cv2.cvtColor(bgr_doc, cv2.COLOR_BGR2GRAY)
    out["gray"] = gray

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(gray)
    out["clahe"] = cl

    dn = cv2.fastNlMeansDenoising(cl, None, h=10, templateWindowSize=7, searchWindowSize=21)
    out["denoise"] = dn

    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
    sh = cv2.filter2D(dn, -1, kernel)
    out["sharpen"] = sh

    th = cv2.adaptiveThreshold(
        sh,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        35,
        10,
    )
    out["thresh"] = th

    return out


pp = preprocess_for_ocr(warped)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (name, im) in zip(axes, list(pp.items())[:4]):
    ax.imshow(im, cmap="gray")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
plt.imshow(pp["thresh"], cmap="gray")
plt.axis("off")
plt.title("Preprocessed (adaptive threshold)")
plt.show()


In [ ]:
# --- Field Localization (template ROIs on normalized warped doc) ---
# Assumption: warped width is fixed at 1000; height varies proportionally.
# These ROIs are tuned for the DEMO front-side layout shown in your sample.


@dataclass
class ROI:
    name: str
    x1: float
    y1: float
    x2: float
    y2: float


def scale_rois(rois: list[ROI], w: int, h: int) -> list[tuple[str, tuple[int, int, int, int]]]:
    out: list[tuple[str, tuple[int, int, int, int]]] = []
    for r in rois:
        x1 = int(r.x1 * w)
        y1 = int(r.y1 * h)
        x2 = int(r.x2 * w)
        y2 = int(r.y2 * h)
        out.append((r.name, (x1, y1, x2, y2)))
    return out


front_rois = [
    # DEMO front layout: narrow value bands to avoid bleeding into adjacent rows.
    ROI("full_name", 0.510, 0.215, 0.925, 0.260),
    ROI("id_number", 0.510, 0.285, 0.925, 0.330),
    ROI("date_of_birth", 0.510, 0.355, 0.725, 0.400),
    ROI("sex", 0.510, 0.425, 0.635, 0.470),
    ROI("nationality", 0.510, 0.495, 0.775, 0.540),
    ROI("place_of_origin", 0.510, 0.565, 0.925, 0.610),
    ROI("place_of_residence", 0.510, 0.630, 0.965, 0.690),
    ROI("issue_date", 0.510, 0.700, 0.710, 0.745),
    ROI("expiry_date", 0.510, 0.765, 0.710, 0.810),
]

h, w = warped.shape[:2]
scaled = scale_rois(front_rois, w=w, h=h)

# Store ROI crops for OCR (using CLAHE grayscale by default)
roi_boxes: dict[str, tuple[int, int, int, int]] = {name: box for name, box in scaled}
roi_for_ocr: dict[str, np.ndarray] = {}
for name, (x1, y1, x2, y2) in scaled:
    roi_for_ocr[name] = pp["clahe"][y1:y2, x1:x2]

vis = warped.copy()
for name, (x1, y1, x2, y2) in scaled:
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(
        vis,
        name,
        (x1, max(0, y1 - 6)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1,
        cv2.LINE_AA,
    )

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Field ROIs (template)")
plt.show()

# Preview a few ROI crops
to_show = ["full_name", "id_number", "date_of_birth"]
fig, axes = plt.subplots(1, len(to_show), figsize=(16, 4))
for ax, key in zip(axes, to_show):
    crop = roi_for_ocr[key]
    ax.imshow(crop, cmap="gray")
    ax.set_title(key)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# --- OCR fields (label-anchored first, ROI fallback) ---
import numpy as np
import pandas as pd
import re
import unicodedata
from dataclasses import dataclass
from difflib import SequenceMatcher

from paddleocr import PaddleOCR

OCR_LANG = "vi"  # Vietnamese model improves accents such as Việt Nam, Bà, Lê

# Init once (CPU)
# PaddleOCR 3.5+: use_textline_orientation replaces use_angle_cls; show_log removed.
ocr = PaddleOCR(lang=OCR_LANG, use_textline_orientation=False)


def _normalize_rec_only_result(res) -> list[tuple[str, float]]:
    # Return list of (text, score) from PaddleOCR output (handles version differences).
    if res is None:
        return []

    # Common shapes:
    # - det=False: [["TEXT", 0.98], ["TEXT2", 0.87]]
    # - sometimes nested: [ [["TEXT", 0.98], ...] ]
    if (
        isinstance(res, list)
        and len(res) == 1
        and isinstance(res[0], list)
        and res[0]
        and isinstance(res[0][0], (list, tuple))
    ):
        res = res[0]

    out: list[tuple[str, float]] = []
    if not isinstance(res, list):
        return out

    for item in res:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            text = str(item[0])
            try:
                score = float(item[1])
            except Exception:
                score = float("nan")
            out.append((text, score))
        elif isinstance(item, str):
            out.append((item, float("nan")))
    return out


def ocr_predict_text(img: np.ndarray) -> tuple[str, float | None, int]:
    # PaddleOCR 3.5 uses predict() -> list[OCRResult]
    bgr_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img
    results = ocr.predict(
        bgr_img,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    if not results:
        return "", None, 0

    j = getattr(results[0], "json", None)
    if not isinstance(j, dict):
        return "", None, 0

    res = j.get("res", {})
    texts = res.get("rec_texts", []) or []
    scores = res.get("rec_scores", []) or []

    text = " ".join([str(t) for t in texts]).strip()
    score_vals = [float(s) for s in scores if s is not None]
    mean_score = float(np.mean(score_vals)) if score_vals else None
    return text, mean_score, len(texts)


VIETNAMESE_TOKEN_CORRECTIONS = {
    "ba": "Bà",
    "bà": "Bà",
    "an": "An",
    "le": "Lê",
    "lé": "Lê",
    "lê": "Lê",
    "viet": "Việt",
    "viét": "Việt",
    "việt": "Việt",
    "nam": "Nam",
    "nu": "Nữ",
    "nữ": "Nữ",
    "duong": "Đường",
    "đuong": "Đường",
    "đường": "Đường",
    "thi": "Thị",
    "thị": "Thị",
    "xa": "xã",
    "xã": "xã",
    "lang": "Láng",
    "láng": "Láng",
    "phuong": "Phường",
    "phường": "Phường",
}


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", str(text).strip())
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[|l]{2,}", "I", text)
    text = re.sub(r"[`']", "", text)
    return text.strip()


def _accentless_key(text: str) -> str:
    decomposed = unicodedata.normalize("NFD", text)
    no_marks = "".join(ch for ch in decomposed if unicodedata.category(ch) != "Mn")
    return no_marks.replace("đ", "d").replace("Đ", "D").lower()


def _correct_tokens(text: str) -> str:
    parts = re.split(r"(\W+)", text)
    corrected: list[str] = []
    for part in parts:
        if not part or re.fullmatch(r"\W+", part):
            corrected.append(part)
            continue
        key = part.lower()
        accentless = _accentless_key(part)
        replacement = VIETNAMESE_TOKEN_CORRECTIONS.get(key) or VIETNAMESE_TOKEN_CORRECTIONS.get(accentless)
        corrected.append(replacement if replacement else part)
    return "".join(corrected)


def correct_vietnamese_ocr_text(field_name: str, text: str) -> str:
    text = clean_text(text)
    if not text:
        return ""

    compact = _accentless_key(text)
    if field_name == "nationality" and "viet" in compact and "nam" in compact:
        return "Việt Nam"
    if field_name == "sex":
        if re.search(r"\bnam\b", compact):
            return "Nam"
        if re.search(r"\bnu\b", compact):
            return "Nữ"
    if field_name in {"full_name", "place_of_origin", "place_of_residence"}:
        text = _correct_tokens(text)
        if field_name == "full_name":
            return " ".join(token[:1].upper() + token[1:] for token in text.split())
        return re.sub(r"\s+", " ", text).strip()
    return text


def is_plausible_field_value(field_name: str, text: str) -> bool:
    text = clean_text(text)
    if not text:
        return False
    if field_name == "id_number":
        return bool(re.search(r"(?:demo[-\s]*)?\d{6,}", text, flags=re.IGNORECASE))
    if field_name in {"date_of_birth", "issue_date", "expiry_date"}:
        return bool(re.search(r"\d{1,2}[/\-.]\d{1,2}[/\-.]\d{4}", text))
    if field_name in {"full_name", "place_of_origin", "place_of_residence"}:
        return len(text) >= 2
    return True


@dataclass
class OCRTextBox:
    text: str
    score: float
    x1: float
    y1: float
    x2: float
    y2: float

    @property
    def cx(self) -> float:
        return (self.x1 + self.x2) / 2.0

    @property
    def cy(self) -> float:
        return (self.y1 + self.y2) / 2.0

    @property
    def h(self) -> float:
        return max(1.0, self.y2 - self.y1)


def _box_to_xyxy(box) -> tuple[float, float, float, float] | None:
    arr = np.asarray(box, dtype=np.float32)
    if arr.size == 4 and arr.ndim == 1:
        x1, y1, x2, y2 = arr.tolist()
        return float(x1), float(y1), float(x2), float(y2)

    if arr.ndim >= 2 and arr.shape[-1] >= 2:
        pts = arr.reshape(-1, arr.shape[-1])[:, :2]
        return (
            float(np.min(pts[:, 0])),
            float(np.min(pts[:, 1])),
            float(np.max(pts[:, 0])),
            float(np.max(pts[:, 1])),
        )

    return None


def _first_nonempty_ocr_boxes(res: dict):
    for key in ("rec_boxes", "rec_polys", "dt_boxes", "dt_polys"):
        boxes = res.get(key)
        if boxes is None:
            continue
        try:
            if len(boxes) > 0:
                return boxes
        except TypeError:
            continue
    return []


def ocr_predict_text_boxes(img: np.ndarray) -> list[OCRTextBox]:
    bgr_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img
    results = ocr.predict(
        bgr_img,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    if not results:
        return []

    j = getattr(results[0], "json", None)
    if not isinstance(j, dict):
        return []

    res = j.get("res", {})
    texts = res.get("rec_texts", []) or []
    scores = res.get("rec_scores", []) or []
    boxes = _first_nonempty_ocr_boxes(res)

    out: list[OCRTextBox] = []
    for i, text in enumerate(texts):
        if i >= len(boxes):
            continue
        xyxy = _box_to_xyxy(boxes[i])
        if xyxy is None:
            continue

        score = float(scores[i]) if i < len(scores) and scores[i] is not None else 0.0
        x1, y1, x2, y2 = xyxy
        clean = str(text).strip()
        if clean:
            out.append(OCRTextBox(clean, score, x1, y1, x2, y2))

    return out


FIELD_LABELS = {
    "full_name": ["FULL NAME"],
    "id_number": ["DEMO ID NO", "ID NO"],
    "date_of_birth": ["DATE OF BIRTH", "BIRTH"],
    "sex": ["SEX"],
    "nationality": ["NATIONALITY"],
    "place_of_origin": ["PLACE OF ORIGIN", "ORIGIN"],
    "place_of_residence": ["PLACE OF RESIDENCE", "RESIDENCE"],
    "issue_date": ["ISSUE DATE"],
    "expiry_date": ["EXPIRY DATE", "EXPIRY"],
}


def _match_key(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", text.lower())


def _label_similarity(text: str, label: str) -> float:
    text_key = _match_key(text)
    label_key = _match_key(label)
    if not text_key or not label_key:
        return 0.0
    if label_key in text_key:
        return 1.0
    return SequenceMatcher(None, text_key, label_key).ratio()


def _is_label_text(text: str) -> bool:
    return any(_label_similarity(text, label) >= 0.78 for labels in FIELD_LABELS.values() for label in labels)


def _find_label_box(text_boxes: list[OCRTextBox], field_name: str) -> OCRTextBox | None:
    best = None
    best_score = 0.0
    for box in text_boxes:
        score = max(_label_similarity(box.text, label) for label in FIELD_LABELS[field_name])
        if score > best_score:
            best = box
            best_score = score

    return best if best is not None and best_score >= 0.68 else None


def _vertical_overlap(a: OCRTextBox, b: OCRTextBox) -> float:
    overlap = max(0.0, min(a.y2, b.y2) - max(a.y1, b.y1))
    return overlap / max(1.0, min(a.h, b.h))


def extract_fields_by_label_anchors(text_boxes: list[OCRTextBox], image_shape: tuple[int, int, int]) -> tuple[dict[str, str], dict[str, float]]:
    if not text_boxes:
        return {}, {}

    h, w = image_shape[:2]
    median_h = float(np.median([b.h for b in text_boxes])) if text_boxes else 20.0
    raw_texts: dict[str, str] = {}
    conf_scores: dict[str, float] = {}

    for field_name in FIELD_LABELS:
        label_box = _find_label_box(text_boxes, field_name)
        if label_box is None:
            continue

        candidates: list[OCRTextBox] = []
        for box in text_boxes:
            if box is label_box or _is_label_text(box.text):
                continue
            if box.x1 < label_box.x2 - 8:
                continue
            same_row = _vertical_overlap(label_box, box) >= 0.20
            near_row = abs(box.cy - label_box.cy) <= max(label_box.h, box.h, median_h) * 0.85
            if same_row or near_row:
                candidates.append(box)

        if not candidates:
            continue

        candidates = sorted(candidates, key=lambda b: (b.x1, b.y1))
        if field_name == "id_number":
            id_candidates = [
                b for b in candidates
                if re.search(r"(?:demo[-\s]*)?\d{6,}", b.text, flags=re.IGNORECASE)
            ]
            if not id_candidates:
                continue
            candidates = id_candidates[:1]
        elif field_name in {"date_of_birth", "issue_date", "expiry_date"}:
            date_candidates = [b for b in candidates if re.search(r"\d{1,2}[/\-.]\d{1,2}[/\-.]\d{4}", b.text)]
            if not date_candidates:
                continue
            candidates = date_candidates[:1]
        else:
            candidates = [b for b in candidates if abs(b.cy - label_box.cy) <= max(label_box.h, b.h, median_h) * 1.10]

        text = " ".join(b.text for b in candidates).strip()
        if text:
            raw_texts[field_name] = text
            conf_scores[field_name] = float(np.mean([b.score for b in candidates]))

    return raw_texts, conf_scores


rows = []
source_by_roi = {}
for roi_name, roi_img in roi_for_ocr.items():
    text, mean_score, n_lines = ocr_predict_text(roi_img)
    rows.append({
        "roi": roi_name,
        "text": text,
        "corrected_text": correct_vietnamese_ocr_text(roi_name, text),
        "mean_score": mean_score,
        "n_lines": n_lines,
        "source": "roi_fallback",
    })
    source_by_roi[roi_name] = len(rows) - 1

text_boxes = ocr_predict_text_boxes(warped)
anchored_texts, anchored_scores = extract_fields_by_label_anchors(text_boxes, warped.shape)
for roi_name, text in anchored_texts.items():
    if roi_name in source_by_roi:
        idx = source_by_roi[roi_name]
        fallback_text = rows[idx]["text"]
        if not is_plausible_field_value(roi_name, text):
            continue
        if is_plausible_field_value(roi_name, fallback_text):
            if roi_name in {"full_name", "id_number", "date_of_birth", "issue_date", "expiry_date"}:
                if len(clean_text(text)) < len(clean_text(fallback_text)):
                    continue
        rows[idx].update({
            "text": text,
            "corrected_text": correct_vietnamese_ocr_text(roi_name, text),
            "mean_score": anchored_scores.get(roi_name),
            "n_lines": 1,
            "source": "label_anchor",
        })

df = pd.DataFrame(rows).sort_values("roi")
df


## Next step (OCR)

- Cell bên trên đã demo OCR từng ROI bằng PaddleOCR (rec-only).
- Bước tiếp theo: postprocess/parse/validate (ngày tháng, số ID, chuẩn hoá Unicode) + dedup/rules + load DB.

Gợi ý: nếu text chưa tốt, thử đổi ROI input sang `pp["gray"]` hoặc tune preprocess theo từng ROI.
